In [1]:
# Celda 1: Importar librerías
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Agregar carpeta raíz al path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.database import execute_query
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, roc_curve
)
from xgboost import XGBClassifier

# Configuración de gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías cargadas correctamente")

✅ Librerías cargadas correctamente


In [2]:
# Celda 2: Cargar dataset INCORRECTO (con data leakage)
print("=" * 70)
print("📊 CARGANDO DATASET INCORRECTO (con data leakage)")
print("=" * 70)

query_incorrecto = """
SELECT *
FROM dbo.dm_dataset_abandono
WHERE es_abandono IS NOT NULL;
"""
df_incorrecto = execute_query(query_incorrecto)

print(f"\n✅ Dataset incorrecto: {df_incorrecto.shape[0]:,} registros")

# Cargar dataset CORRECTO (sin data leakage)
print("\n" + "=" * 70)
print("📊 CARGANDO DATASET CORRECTO (sin data leakage)")
print("=" * 70)

query_correcto = """
SELECT *
FROM dbo.dm_dataset_abandono_correcto
WHERE es_abandono IS NOT NULL;
"""
df_correcto = execute_query(query_correcto)

print(f"\n✅ Dataset correcto: {df_correcto.shape[0]:,} registros")

# Comparar dimensiones
print("\n" + "=" * 70)
print("📋 COMPARACIÓN DE DIMENSIONES")
print("=" * 70)
print(f"Dataset incorrecto: {df_incorrecto.shape[0]:,} filas × {df_incorrecto.shape[1]} columnas")
print(f"Dataset correcto:   {df_correcto.shape[0]:,} filas × {df_correcto.shape[1]} columnas")

📊 CARGANDO DATASET INCORRECTO (con data leakage)
❌ Error de conexión a SQL Server:
   ('HYT00', '[HYT00] [Microsoft][ODBC Driver 18 for SQL Server]Login timeout expired (0) (SQLDriverConnect)')

💡 Posibles soluciones:
   1. Verifica que SQL Server esté corriendo
   2. Revisa las credenciales en el archivo .env
   3. Asegúrate de que la base 'SkyTravelDW' exista
   4. Verifica el driver ODBC instalado


AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
# Celda 3: Función reutilizable para entrenar modelo
def entrenar_y_evaluar(df, nombre_dataset):
    """
    Entrena un modelo XGBoost y retorna métricas de evaluación.
    """
    print(f"\n{'=' * 70}")
    print(f"🤖 ENTRENANDO MODELO: {nombre_dataset}")
    print(f"{'=' * 70}")
    
    # 1. Preparar variables
    columnas_a_eliminar = ['id_reserva', 'id_cliente', 'fecha_inicio_reserva', 'es_abandono']
    X = df.drop(columns=columnas_a_eliminar, errors='ignore')
    y = df['es_abandono']
    
    # 2. Codificar variables categóricas
    columnas_texto = X.select_dtypes(include=['object']).columns
    if len(columnas_texto) > 0:
        X = pd.get_dummies(X, columns=columnas_texto.tolist(), drop_first=True)
    
    # 3. Convertir booleanos a enteros
    X = X.astype({col: 'int8' for col in X.select_dtypes(include=['bool']).columns})
    
    # 4. Rellenar nulos
    X = X.fillna(0)
    
    # 5. Dividir train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    
    print(f"📏 Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
    print(f"📊 Features: {X.shape[1]}")
    
    # 6. Entrenar modelo
    ratio_desbalance = (y_train == 0).sum() / (y_train == 1).sum()
    
    modelo = XGBClassifier(
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=ratio_desbalance,
        random_state=42,
        eval_metric='auc',
        verbosity=0  # Silenciar warnings
    )
    
    modelo.fit(X_train, y_train)
    
    # 7. Evaluar
    y_pred = modelo.predict(X_test)
    y_pred_proba = modelo.predict_proba(X_test)[:, 1]
    
    accuracy = (y_pred == y_test).mean()
    auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\n📊 MÉTRICAS:")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   ROC-AUC:  {auc:.4f}")
    
    # 8. Feature importance
    importancia = pd.DataFrame({
        'Variable': X.columns,
        'Importancia': modelo.feature_importances_
    }).sort_values(by='Importancia', ascending=False)
    
    return {
        'modelo': modelo,
        'X_test': X_test,
        'y_test': y_test,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'accuracy': accuracy,
        'auc': auc,
        'importancia': importancia,
        'nombre': nombre_dataset
    }

In [ ]:
# Celda 4: Entrenar ambos modelos
resultados_incorrecto = entrenar_y_evaluar(df_incorrecto, "INCORRECTO (con leakage)")
resultados_correcto = entrenar_y_evaluar(df_correcto, "CORRECTO (sin leakage)")

In [ ]:
# Celda 5: Comparación visual de métricas
print("=" * 70)
print("📊 COMPARACIÓN DE MÉTRICAS")
print("=" * 70)

comparacion = pd.DataFrame({
    'Métrica': ['Accuracy', 'ROC-AUC'],
    'Incorrecto (con leakage)': [
        f"{resultados_incorrecto['accuracy']:.4f}",
        f"{resultados_incorrecto['auc']:.4f}"
    ],
    'Correcto (sin leakage)': [
        f"{resultados_correcto['accuracy']:.4f}",
        f"{resultados_correcto['auc']:.4f}"
    ]
})

print(comparacion.to_string(index=False))

# Gráfico de barras comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].bar(['Incorrecto', 'Correcto'], 
            [resultados_incorrecto['accuracy'], resultados_correcto['accuracy']],
            color=['#e74c3c', '#2ecc71'])
axes[0].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Valor')
axes[0].set_ylim(0, 1.05)
for i, v in enumerate([resultados_incorrecto['accuracy'], resultados_correcto['accuracy']]):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# ROC-AUC
axes[1].bar(['Incorrecto', 'Correcto'], 
            [resultados_incorrecto['auc'], resultados_correcto['auc']],
            color=['#e74c3c', '#2ecc71'])
axes[1].set_title('ROC-AUC', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Valor')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate([resultados_incorrecto['auc'], resultados_correcto['auc']]):
    axes[1].text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("🔍 INTERPRETACIÓN:")
print("=" * 70)
print(f"• Modelo INCORRECTO: AUC = {resultados_incorrecto['auc']:.4f} (sospechosamente perfecto)")
print(f"• Modelo CORRECTO:   AUC = {resultados_correcto['auc']:.4f} (realista y útil)")
print(f"\n⚠️  El modelo incorrecto tiene data leakage: usa información del futuro")
print(f"✅ El modelo correcto solo usa información disponible al momento de la reserva")

In [ ]:
# Celda 6: Comparar las variables más importantes
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 10 Incorrecto
top10_incorrecto = resultados_incorrecto['importancia'].head(10)
axes[0].barh(top10_incorrecto['Variable'][::-1], 
             top10_incorrecto['Importancia'][::-1],
             color='#e74c3c')
axes[0].set_title('Modelo INCORRECTO (con leakage)\nTop 10 Variables', 
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia')

# Top 10 Correcto
top10_correcto = resultados_correcto['importancia'].head(10)
axes[1].barh(top10_correcto['Variable'][::-1], 
             top10_correcto['Importancia'][::-1],
             color='#2ecc71')
axes[1].set_title('Modelo CORRECTO (sin leakage)\nTop 10 Variables', 
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.show()

# Tabla comparativa
print("=" * 70)
print("📋 TOP 10 VARIABLES - COMPARACIÓN")
print("=" * 70)

comparacion_vars = pd.DataFrame({
    'Rank Incorrecto': range(1, 11),
    'Variable Incorrecta': top10_incorrecto['Variable'].values,
    'Importancia Incorrecta': top10_incorrecto['Importancia'].values.round(4),
    'Rank Correcto': range(1, 11),
    'Variable Correcta': top10_correcto['Variable'].values,
    'Importancia Correcta': top10_correcto['Importancia'].values.round(4)
})

print(comparacion_vars.to_string(index=False))

In [ ]:
# Celda 7: Curvas ROC lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Curva ROC Incorrecto
fpr_inc, tpr_inc, _ = roc_curve(resultados_incorrecto['y_test'], 
                                 resultados_incorrecto['y_pred_proba'])
axes[0].plot(fpr_inc, tpr_inc, color='#e74c3c', lw=2, 
             label=f'ROC (AUC = {resultados_incorrecto["auc"]:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Modelo INCORRECTO\n(con data leakage)', fontweight='bold')
axes[0].legend(loc="lower right")

# Curva ROC Correcto
fpr_cor, tpr_cor, _ = roc_curve(resultados_correcto['y_test'], 
                                 resultados_correcto['y_pred_proba'])
axes[1].plot(fpr_cor, tpr_cor, color='#2ecc71', lw=2, 
             label=f'ROC (AUC = {resultados_correcto["auc"]:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Modelo CORRECTO\n(sin data leakage)', fontweight='bold')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
# Celda 8: Reporte ejecutivo
print("=" * 70)
print("📊 REPORTE EJECUTIVO: IMPACTO DEL DATA LEAKAGE")
print("=" * 70)

print(f"""
🔴 MODELO INCORRECTO (con data leakage):
   • Accuracy: {resultados_incorrecto['accuracy']:.2%}
   • ROC-AUC:  {resultados_incorrecto['auc']:.4f}
   • Variable #1: {resultados_incorrecto['importancia'].iloc[0]['Variable']}
   • Variable #2: {resultados_incorrecto['importancia'].iloc[1]['Variable']}
   • Variable #3: {resultados_incorrecto['importancia'].iloc[2]['Variable']}
   
   ⚠️  PROBLEMA: Usa información del futuro (monto_total_12m, compras_previas_12m)
   ⚠️  RESULTADO: Métricas perfectas pero modelo INÚTIL en producción

🟢 MODELO CORRECTO (sin data leakage):
   • Accuracy: {resultados_correcto['accuracy']:.2%}
   • ROC-AUC:  {resultados_correcto['auc']:.4f}
   • Variable #1: {resultados_correcto['importancia'].iloc[0]['Variable']}
   • Variable #2: {resultados_correcto['importancia'].iloc[1]['Variable']}
   • Variable #3: {resultados_correcto['importancia'].iloc[2]['Variable']}
   
   ✅ VENTAJA: Solo usa información disponible al momento de la reserva
   ✅ RESULTADO: Métricas realistas y modelo ÚTIL en producción

💡 CONCLUSIÓN:
   Un modelo con AUC = 1.0 es SOSPECHOSO. En datos reales, los mejores
   modelos de abandono tienen AUC entre 0.75 y 0.88.
   
   El modelo correcto (AUC ≈ 0.80) es más valioso porque:
   1. Es honesto sobre lo que puede predecir
   2. Funciona en producción con datos reales
   3. Genera insights accionables para el negocio
""")

print("=" * 70)
print("✅ ANÁLISIS COMPLETADO")
print("=" * 70)